# SOTERAI ML LAKERA-COMPARISON TRAINING COLAB

CUDA GPU-only training notebook for the SoterAI security classifier. TPU support is not implemented. This notebook keeps Colab's native Torch/CUDA stack intact, installs pinned ML libraries, accepts a versioned training bundle, verifies a reviewed `FINAL_LOCKED` split, and persists candidate artifacts to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python --version
!nvidia-smi || true

In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile

REPO_DIR = pathlib.Path('/content/soterai')
FORCE_REEXTRACT = True

def repair_windows_zip_paths(root):
    repaired = 0
    for path in list(root.rglob('*')):
        if not path.is_file() or '\\' not in path.name:
            continue
        parts = path.name.split('\\')
        target = path.parent.joinpath(*parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(path), str(target))
        repaired += 1
    if repaired:
        print('repaired windows-style zip entries:', repaired)

def find_repo_root(root):
    for package_json in root.rglob('package.json'):
        candidate = package_json.parent
        if (candidate / 'scripts' / 'ml' / 'soterai_training_pipeline.py').exists():
            return candidate
    if (root / 'scripts' / 'ml' / 'soterai_training_pipeline.py').exists():
        return root
    return None

if FORCE_REEXTRACT and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    candidates = sorted(pathlib.Path('/content').glob('soterai-train-bundle*.zip'))
    if not candidates:
        from google.colab import files
        print('Upload soterai-train-bundle.zip from your project folder.')
        uploaded = files.upload()
        candidates = [pathlib.Path('/content') / name for name in uploaded if name.endswith('.zip')]
    if not candidates:
        raise RuntimeError('No training bundle zip was uploaded.')
    zip_path = max(candidates, key=lambda p: p.stat().st_mtime)
    extract_dir = pathlib.Path('/content/soterai_extract')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    print('extracting', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    repair_windows_zip_paths(extract_dir)
    source_root = find_repo_root(extract_dir)
    if source_root is None:
        raise RuntimeError('Bundle does not contain SoterAI training files.')
    shutil.move(str(source_root), str(REPO_DIR))

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'scripts' / 'ml'))
print('repo', REPO_DIR)
try:
    print('commit', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
except Exception:
    print('commit unavailable in uploaded bundle')
print('datasets', [p.name for p in sorted((REPO_DIR / 'datasets').glob('*.jsonl'))[:20]])

In [ ]:
# Do not install/downgrade torch or numpy here. Colab already provides the CUDA-matched build.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--upgrade-strategy', 'only-if-needed',
    '-r', 'requirements-colab.txt',
], check=True)

import google.protobuf, torch, transformers, sklearn, numpy
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('protobuf', google.protobuf.__version__)
print('sklearn', sklearn.__version__, 'numpy', numpy.__version__)

In [ ]:
from soterai_training_pipeline import require_accelerator
accelerator = require_accelerator(smoke_only=False)

In [ ]:
DATASET = '/content/soterai/datasets/ml-augmented-v6.jsonl'
RUN_ROOT = '/content/drive/MyDrive/soterai-ml-runs'
FREEZE_DIR = '/content/drive/MyDrive/soterai-ml-freezes/v1'
SPLIT_FREEZE = f'{FREEZE_DIR}/split-freeze.json'
GENERATE_PROVISIONAL_FREEZE_IF_MISSING = True
BASE_MODEL = 'microsoft/deberta-v3-base'
EPOCHS = 4
BATCH_SIZE = 8
GRAD_ACCUM = 8
MAX_LENGTH = 256
LEARNING_RATE = '2e-5'
SEED = 42

import pathlib
if not pathlib.Path(DATASET).exists():
    raise RuntimeError(f'Dataset missing: {DATASET}')
print('training dataset', DATASET)

In [ ]:
import json, subprocess, sys
subprocess.run([
    sys.executable, 'scripts/ml/soterai_dataset_audit.py', DATASET,
    '--out', f'{RUN_ROOT}/dataset-audit.json',
], check=True)

In [ ]:
# P0 gate: a reviewed freeze must exist before fitting. The local builder only
# creates PROVISIONAL output and therefore cannot self-promote a dataset.
if not pathlib.Path(SPLIT_FREEZE).is_file() and GENERATE_PROVISIONAL_FREEZE_IF_MISSING:
    subprocess.run([
        sys.executable, "scripts/ml/soterai_data_freeze.py",
        "--dataset", DATASET,
        "--output-dir", FREEZE_DIR,
        "--taxonomy-version", "SOTERAI-ML-TAXONOMY-v1",
        "--seed", str(SEED),
    ], check=True)
if not pathlib.Path(SPLIT_FREEZE).is_file():
    raise RuntimeError(f'Reviewed split freeze missing: {SPLIT_FREEZE}')
freeze = json.load(open(SPLIT_FREEZE, encoding="utf-8"))
audit = json.load(open(f"{FREEZE_DIR}/dataset-forensic-audit.json", encoding="utf-8"))
if freeze.get("status") != "FINAL_LOCKED" or audit.get("status") != "FINAL_LOCKED" or not audit.get("coverage_gate", {}).get("passed", False):
    raise RuntimeError(
        "Training blocked before model fitting: the split is not FINAL_LOCKED or failed coverage. "
        + json.dumps(audit.get("coverage_gate", {}), ensure_ascii=False)
    )
if not audit["lexical_near_duplicate_audit"].get("semantic_embedding_audit_complete", False):
    raise RuntimeError("Training blocked: accelerator-based semantic leakage audit is incomplete.")


In [ ]:
import glob, subprocess, sys
cmd = [
    sys.executable, '-u', 'scripts/ml/soterai_colab_train_runner.py',
    '--dataset', DATASET,
    '--split-freeze', SPLIT_FREEZE,
    '--output-root', RUN_ROOT,
    '--seed', str(SEED),
]
print('running:', ' '.join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout)
print('training_runner_exit_code:', proc.returncode)
if proc.returncode != 0:
    logs = sorted(glob.glob(RUN_ROOT + '/_training-logs/*.log'))
    if logs:
        print('latest training log:', logs[-1])
        print(open(logs[-1], encoding='utf-8').read()[-12000:])
    raise RuntimeError('SoterAI training failed. The real error is printed above from the training log.')

In [ ]:
import glob, json
summaries = sorted(glob.glob(RUN_ROOT + '/*/experiment_summary.json'))
if not summaries:
    logs = sorted(glob.glob(RUN_ROOT + '/_training-logs/*.log'))
    if logs:
        print('latest training log:', logs[-1])
        print(open(logs[-1], encoding='utf-8').read()[-8000:])
    raise RuntimeError('No experiment summary was produced. The training cell above failed or was not run; use its traceback, not this summary cell, as the real error.')
latest = summaries[-1]
print('latest summary:', latest)
summary = json.load(open(latest))
print(json.dumps({
    'experiment_id': summary['experiment_id'],
    'artifact_dir': summary['artifact_dir'],
    'best_epoch': summary['best_epoch'],
    'splits': summary['splits'],
    'test_accuracy': summary['test_metrics']['accuracy'],
    'test_f1_macro': summary['test_metrics']['f1_macro'],
    'test_f1_weighted': summary['test_metrics']['f1_weighted'],
}, indent=2))